# Data Science in Banking — A Workshop for C# .NET Developers

Welcome! In this workshop you'll step into the shoes of a **data science team** in the banking sector.
We'll use the same banking database your applications would serve in production — customers, accounts, cards, and transactions — to explore, visualize, and even predict with Python's data science stack.

## What We'll Cover
| # | Topic | Library | C# Analogy |
|---|-------|---------|-------------|
| 1 | **Loading & Exploring Data** | `pandas` + `sqlite3` | Entity Framework + LINQ |
| 2 | **Data Wrangling & Aggregation** | `pandas` | LINQ GroupBy / Select |
| 3 | **Data Visualization** | `matplotlib` | LiveCharts / OxyPlot |
| 4 | **Simple Machine Learning** | `scikit-learn` | ML.NET |

> **Prerequisites:** The banking database at `data/banking_data.db` must already be generated.
> Run `python generate_banking_db.py` from the `database-admin/` folder if you haven't done so yet.

---

## 0. Setup — Import Libraries

In C# you'd write `using System.Data.SqlClient;` — in Python we import packages the same way.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make plots look nicer
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Display wider tables in the notebook
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

print('Libraries loaded ✅')

## 1. Loading & Exploring Data with Pandas

### 1.1 Connecting to the Database

Think of `pandas.read_sql()` like calling `DbContext.Set<Customer>().ToList()` —
it runs a SQL query and returns the results as a **DataFrame** (Python's version of a typed `DataTable`).

In [ ]:
# Connect to the SQLite banking database
# C# equivalent: new SqliteConnection("Data Source=banking_data.db")
conn = sqlite3.connect('data/banking_data.db')

# Load each table into a pandas DataFrame
customers = pd.read_sql('SELECT * FROM customers', conn)
accounts = pd.read_sql('SELECT * FROM accounts', conn)
cards = pd.read_sql('SELECT * FROM cards', conn)
transactions = pd.read_sql('SELECT * FROM transactions', conn)

print(f'Customers:    {len(customers):>6,} rows')
print(f'Accounts:     {len(accounts):>6,} rows')
print(f'Cards:        {len(cards):>6,} rows')
print(f'Transactions: {len(transactions):>6,} rows')

### 1.2 First Look at a DataFrame

A DataFrame is like a `List<Dictionary<string, object>>` with super-powers.

In [ ]:
# .head() — like .Take(5) in LINQ
customers.head()

In [ ]:
# .info() — schema overview (column names, types, nulls)
# Think of it as inspecting your EF Core model properties
customers.info()

In [ ]:
# .describe() — instant statistical summary for every numeric column
# No C# one-liner equivalent — you'd need to write this manually
accounts.describe()

### 1.3 Quick Data Quality Checks

In [ ]:
# Check for null values — like checking entity.Property == null across all rows
print('Null values per column in transactions:')
print(transactions.isnull().sum())
print(f'\nTotal rows: {len(transactions)}')

In [ ]:
# Unique value counts — like .GroupBy(x => x.Status).Select(g => new { g.Key, Count = g.Count() })
print('Customer statuses:')
print(customers['status'].value_counts())

print('\nAccount types:')
print(accounts['account_type'].value_counts())

print('\nTransaction types:')
print(transactions['type'].value_counts())

---
## 2. Data Wrangling & Aggregation

This is where pandas truly shines — think of it as **LINQ on steroids** for tabular data.

### 2.1 Filtering Rows

In [ ]:
# Filter active accounts with balance > 10,000
# C# LINQ: accounts.Where(a => a.Status == "active" && a.Balance > 10000)
high_balance = accounts[(accounts['status'] == 'active') & (accounts['balance'] > 10000)]

print(f'Active accounts with balance > $10,000: {len(high_balance)}')
high_balance[['account_id', 'customer_id', 'account_type', 'balance']].head(10)

### 2.2 Joining DataFrames

In [ ]:
# Join accounts with customers — like a SQL JOIN or LINQ .Join()
# C# LINQ: from a in accounts join c in customers on a.CustomerId equals c.CustomerId
account_customer = accounts.merge(customers, on='customer_id', suffixes=('_account', '_customer'))

# Show a few columns
account_customer[[
    'account_id', 'account_type', 'balance', 'status_account',
    'first_name', 'last_name', 'email'
]].head(10)

### 2.3 GroupBy & Aggregation

In [ ]:
# Total balance per account type
# C# LINQ: accounts.GroupBy(a => a.AccountType)
#                   .Select(g => new { Type = g.Key, Total = g.Sum(a => a.Balance), Count = g.Count() })
balance_by_type = accounts.groupby('account_type').agg(
    total_balance=('balance', 'sum'),
    avg_balance=('balance', 'mean'),
    account_count=('account_id', 'count')
).round(2)

balance_by_type

In [ ]:
# Customer portfolio summary: total balance across all their accounts
customer_portfolio = accounts[accounts['status'] == 'active'].groupby('customer_id').agg(
    num_accounts=('account_id', 'count'),
    total_balance=('balance', 'sum')
).round(2).reset_index()

# Merge with customer names
customer_portfolio = customer_portfolio.merge(
    customers[['customer_id', 'first_name', 'last_name']],
    on='customer_id'
)

# Top 10 wealthiest customers
top_customers = customer_portfolio.sort_values('total_balance', ascending=False).head(10)
top_customers[['first_name', 'last_name', 'num_accounts', 'total_balance']]

### 2.4 Working with Dates

In [ ]:
# Convert string dates to datetime — C# equivalent: DateTime.Parse()
transactions['created_at'] = pd.to_datetime(transactions['created_at'], format='ISO8601')

# Extract useful components
transactions['year_month'] = transactions['created_at'].dt.to_period('M')
transactions['day_of_week'] = transactions['created_at'].dt.day_name()
transactions['hour'] = transactions['created_at'].dt.hour

transactions[['transaction_id', 'created_at', 'year_month', 'day_of_week', 'hour']].head()

In [ ]:
# Monthly transaction volume and total amounts
monthly = transactions.groupby('year_month').agg(
    transaction_count=('transaction_id', 'count'),
    total_amount=('amount', 'sum')
).reset_index()

monthly['year_month'] = monthly['year_month'].astype(str)
monthly.tail(12)

---
## 3. Data Visualization with Matplotlib

In C# you might use **LiveCharts**, **OxyPlot**, or **ScottPlot**.
In Python, `matplotlib` is the standard — and it integrates seamlessly with Jupyter notebooks.

### 3.1 Account Balance Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of balances
active_accounts = accounts[accounts['status'] == 'active']
axes[0].hist(active_accounts['balance'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Distribution of Active Account Balances')
axes[0].set_xlabel('Balance ($)')
axes[0].set_ylabel('Number of Accounts')

# Box plot by account type
active_accounts.boxplot(column='balance', by='account_type', ax=axes[1])
axes[1].set_title('Balance by Account Type')
axes[1].set_xlabel('Account Type')
axes[1].set_ylabel('Balance ($)')
fig.suptitle('')  # Remove auto-title from boxplot

plt.tight_layout()
plt.show()

### 3.2 Transaction Volume Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(monthly['year_month'], monthly['transaction_count'], color='steelblue', alpha=0.8)
ax.set_title('Monthly Transaction Volume')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Transactions')

# Show only every N-th label to avoid crowding
n = max(1, len(monthly) // 12)
ax.set_xticks(range(0, len(monthly), n))
ax.set_xticklabels(monthly['year_month'].iloc[::n], rotation=45, ha='right')

plt.tight_layout()
plt.show()

### 3.3 Transaction Types Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts = transactions['type'].value_counts()
type_amounts = transactions.groupby('type')['amount'].sum()

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

# Pie chart — count
axes[0].pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Transaction Count by Type')

# Pie chart — total amount
axes[1].pie(type_amounts, labels=type_amounts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Total Amount by Transaction Type')

plt.tight_layout()
plt.show()

### 3.4 Top 10 Customers by Portfolio Value

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

top10 = top_customers.head(10).copy()
top10['name'] = top10['first_name'] + ' ' + top10['last_name']

bars = ax.barh(top10['name'], top10['total_balance'], color='steelblue', edgecolor='black')
ax.set_xlabel('Total Balance ($)')
ax.set_title('Top 10 Customers by Total Portfolio Value')
ax.invert_yaxis()  # Highest at the top

# Add value labels
for bar in bars:
    width = bar.get_width()
    ax.text(width + 200, bar.get_y() + bar.get_height() / 2,
            f'${width:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 3.5 Transaction Patterns — Day of Week & Hour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Day of week order
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = transactions['day_of_week'].value_counts().reindex(day_order)

axes[0].bar(day_counts.index, day_counts.values, color='steelblue', edgecolor='black')
axes[0].set_title('Transactions by Day of Week')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Hour of day
hour_counts = transactions['hour'].value_counts().sort_index()

axes[1].plot(hour_counts.index, hour_counts.values, marker='o', color='steelblue', linewidth=2)
axes[1].fill_between(hour_counts.index, hour_counts.values, alpha=0.2, color='steelblue')
axes[1].set_title('Transactions by Hour of Day')
axes[1].set_xlabel('Hour (24h)')
axes[1].set_ylabel('Count')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

---
## 4. Simple Machine Learning with scikit-learn

In C# you'd use **ML.NET** to build a pipeline, train a model, and make predictions.
In Python the dominant library is **scikit-learn** — the API is very consistent:

```python
model = SomeModel()      # Instantiate
model.fit(X_train, y_train)  # Train  (like mlContext.Fit())
predictions = model.predict(X_test)  # Predict (like engine.Predict())
```

### Use Case: Predict Account Type from Transaction Behavior

Given a customer's transaction patterns (number of transactions, average amount, etc.),
can we predict their **account type** (checking / savings / business)?

This is a **classification** problem — exactly the kind of thing a banking data science team would tackle.

### 4.1 Feature Engineering

First we need to build **features** (input variables) from raw transactions.
This is the pandas wrangling from Section 2, now applied for ML.

In [ ]:
# Build features per account from transaction history
account_features = transactions.groupby('account_id').agg(
    txn_count=('transaction_id', 'count'),
    avg_amount=('amount', 'mean'),
    max_amount=('amount', 'max'),
    total_amount=('amount', 'sum'),
    num_debits=('type', lambda x: (x == 'debit').sum()),
    num_credits=('type', lambda x: (x == 'credit').sum()),
    num_transfers=('type', lambda x: (x == 'transfer').sum()),
    num_fees=('type', lambda x: (x == 'fee').sum()),
    final_balance=('balance_after', 'last')
).reset_index()

# Add the label (what we want to predict)
account_features = account_features.merge(
    accounts[['account_id', 'account_type']],
    on='account_id'
)

print(f'Accounts with features: {len(account_features)}')
account_features.head()

### 4.2 Prepare Data for Training

Split into features (`X`) and label (`y`), then train/test split — exactly like ML.NET's `TrainTestData`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Features (X) — all numeric columns we engineered
feature_columns = ['txn_count', 'avg_amount', 'max_amount', 'total_amount',
                   'num_debits', 'num_credits', 'num_transfers', 'num_fees',
                   'final_balance']

X = account_features[feature_columns]

# Label (y) — account type encoded as integers
# C# ML.NET equivalent: MapValueToKey("Label", "account_type")
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(account_features['account_type'])

print('Classes:', label_encoder.classes_)
print(f'Features shape: {X.shape}')
print(f'Label distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# Train / Test split (80% train, 20% test)
# C# ML.NET: mlContext.Data.TrainTestSplit(data, testFraction: 0.2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')

### 4.3 Train a Decision Tree Classifier

A **Decision Tree** is one of the simplest and most interpretable ML models.
It asks a series of yes/no questions about the features ("Is avg_amount > 500?")
to arrive at a prediction — very intuitive for banking rule-based thinking.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Create and train the model
# C# ML.NET equivalent:
#   var pipeline = mlContext.Transforms.Concatenate("Features", featureColumns)
#       .Append(mlContext.MulticlassClassification.Trainers.SdcaMaximumEntropy());
#   var model = pipeline.Fit(trainData);

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

print('Model trained ✅')

### 4.4 Evaluate the Model

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2%}\n')

# Detailed classification report
# Precision, Recall, F1-score — standard ML metrics
print(classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_
))

### 4.5 Feature Importance — What Matters Most?

A key advantage of tree models: we can see **which features the model relied on**.

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot.barh(ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Feature Importance — Decision Tree Classifier')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

### 4.6 Visualize the Decision Tree

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    model,
    feature_names=feature_columns,
    class_names=list(label_encoder.classes_),
    filled=True,
    rounded=True,
    ax=ax,
    max_depth=3,
    fontsize=9
)
ax.set_title('Decision Tree (first 3 levels)', fontsize=14)
plt.tight_layout()
plt.show()

### 4.7 Confusion Matrix — Where Does the Model Get It Wrong?

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

### 4.8 Bonus — Try a Random Forest

A **Random Forest** is an ensemble of many decision trees — like running multiple
independent classifiers and taking a majority vote. Almost always better than a single tree.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f'Decision Tree Accuracy:  {accuracy:.2%}')
print(f'Random Forest Accuracy:  {rf_accuracy:.2%}')
print(f'\nImprovement: {rf_accuracy - accuracy:+.2%}')

print('\nRandom Forest Classification Report:')
print(classification_report(y_test, rf_pred, target_names=label_encoder.classes_))

---
## 5. Summary & Key Takeaways

| What We Did | Python Tool | C# Equivalent |
|-------------|-------------|----------------|
| Connect to DB & load data | `sqlite3` + `pandas.read_sql()` | `SqliteConnection` + EF Core |
| Filter, join, aggregate | `DataFrame` operations | LINQ (`.Where()`, `.Join()`, `.GroupBy()`) |
| Statistical summary | `df.describe()` | Manual calculation or MathNet.Numerics |
| Charts & visualizations | `matplotlib` | LiveCharts / OxyPlot / ScottPlot |
| ML classification | `scikit-learn` | ML.NET |
| Feature engineering | pandas groupby + agg | LINQ + custom classes |

### Why Python for Data Science?

- **Ecosystem**: pandas, matplotlib, scikit-learn, TensorFlow, PyTorch — unmatched breadth
- **Notebooks**: Interactive exploration (this file!) — no compile-run cycle
- **Speed of iteration**: Load data → transform → visualize → model in minutes
- **Community**: Most ML research papers ship Python code

### When to Keep Using C#

- **Production APIs** serving ML models (consider ONNX to bridge the two)
- **Real-time transaction processing** where latency matters
- **Enterprise applications** with complex business logic
- **Type safety** for critical financial calculations

> **Pro tip:** Many teams use *both* — Python for exploration & model training,
> C# for production services that consume the trained models via ONNX or REST APIs.

---
**Happy exploring! 🐍📊🏦**

In [ ]:
# Don't forget to close the database connection
conn.close()
print('Database connection closed ✅')
